# Belka · 04 · LGPD na Ouro
As fontes do pipeline (Open-Meteo e IBGE) **não têm dado pessoal**: são clima por coordenada e população
agregada por município. O dado pessoal da Belka aparece no produto real, no **cadastro de quem recebe alertas**
(telefone para SMS, e-mail, endereço, idade para priorizar idosos).

Para demonstrar a proteção sem expor ninguém, este notebook **gera um cadastro simulado** e mostra o que
chega à Ouro depois da proteção.

| Garantia | Como |
|---|---|
| Nenhum dado real | telefones com DDD **00** (não existe no Brasil), e-mails no domínio reservado **exemplo.com** e CEPs começando em **00** (não existem) |
| Dado bruto separado | cadastro em `workspace.belka_seguro`, schema próprio de acesso restrito, fora da camada analítica |
| Segredo fora do código | o *salt* do hash é sorteado uma vez e guardado em tabela no schema seguro; o valor nunca aparece no código, no repositório nem nos prints |

In [0]:
%sql
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


## 1. Schema seguro e salt do hash
O telefone vai virar um hash SHA-256. Só que existem poucos telefones possíveis (cerca de 1 bilhão de
celulares), então um hash **sem salt** seria revertido por força bruta em minutos. O salt é um valor secreto
e aleatório somado ao telefone antes do hash.

`CREATE TABLE IF NOT EXISTS` sorteia o salt **só na primeira execução**. Nas seguintes ele é mantido, e o mesmo
telefone gera sempre o mesmo hash. Em produção ele ficaria num cofre de segredos (Databricks Secrets, Key Vault).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.belka_seguro
COMMENT 'Belka · segredos e dados pessoais operacionais. Acesso restrito, fora da camada analítica.';

CREATE TABLE IF NOT EXISTS workspace.belka_seguro.config_hash
COMMENT 'Salt do hash de telefone. Nunca consultar o valor em notebook ou print.'
AS SELECT
  'v1' AS versao,
  sha2(concat(uuid(), uuid(), CAST(current_timestamp() AS STRING)), 256) AS salt,
  current_timestamp() AS criado_em;

-- Conferência sem revelar o segredo: só versão, tamanho e data
SELECT versao, length(salt) AS tamanho_salt, criado_em
FROM workspace.belka_seguro.config_hash;

versao,tamanho_salt,criado_em
v1,64,2026-11-07T19:21:27.951-03:00


## 2. Cadastro simulado de assinantes (dado pessoal bruto)
Representa o cadastro do app da Belka: 1.000 assinantes com nome, telefone, e-mail, CEP, município,
data de nascimento, canal de alerta e consentimento. É gerado de forma **determinística** (`xxhash64` do id),
então rodar de novo produz exatamente o mesmo cadastro.
A distribuição por município acompanha, de forma aproximada, o tamanho das cidades.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.belka_seguro.prata_assinante
COMMENT 'Belka · cadastro de assinantes SIMULADO (DDD 00, exemplo.com, CEP 00xxx). Dado pessoal bruto: acesso restrito.'
AS
WITH sorteio AS (
  SELECT
    CAST(id AS INT) AS assinante_id,
    CAST(pmod(xxhash64(id, 'nome'), 20)      AS INT) AS i_nome,
    CAST(pmod(xxhash64(id, 'sobrenome'), 20) AS INT) AS i_sobrenome,
    pmod(xxhash64(id, 'municipio'), 100)             AS p_municipio,
    lpad(CAST(pmod(xxhash64(id, 'telefone'), 100000000) AS STRING), 8, '0') AS n_telefone,
    lpad(CAST(pmod(xxhash64(id, 'cep'), 1000000) AS STRING), 6, '0')        AS n_cep,
    CAST(pmod(xxhash64(id, 'nascimento'), 25000) AS INT) AS n_nascimento,
    pmod(xxhash64(id, 'canal'), 100)          AS p_canal,
    pmod(xxhash64(id, 'consentimento'), 100)  AS p_consentimento,
    CAST(pmod(xxhash64(id, 'data'), 30) AS INT) AS n_data
  FROM range(1, 1001)
),
nomes AS (
  SELECT *,
    element_at(array('Ana','Bruno','Carla','Daniel','Eduarda','Felipe','Gabriela','Henrique','Isabela','João',
                     'Larissa','Marcos','Natália','Otávio','Patrícia','Rafael','Sofia','Thiago','Vanessa','Wagner'),
               i_nome + 1) AS primeiro_nome,
    element_at(array('Silva','Santos','Oliveira','Souza','Lima','Pereira','Costa','Rodrigues','Almeida','Nascimento',
                     'Carvalho','Gomes','Martins','Araújo','Ribeiro','Barbosa','Rocha','Dias','Moreira','Teixeira'),
               i_sobrenome + 1) AS sobrenome
  FROM sorteio
)
SELECT
  assinante_id,
  concat(primeiro_nome, ' ', sobrenome) AS nome,
  concat('(00) 9', substr(n_telefone, 1, 4), '-', substr(n_telefone, 5, 4)) AS telefone,
  lower(concat(
    translate(primeiro_nome, 'áéíóúâêôãõçÁÉÍÓÚÂÊÔÃÕÇ', 'aeiouaeoaocAEIOUAEOAOC'), '.',
    translate(sobrenome,     'áéíóúâêôãõçÁÉÍÓÚÂÊÔÃÕÇ', 'aeiouaeoaocAEIOUAEOAOC'),
    assinante_id, '@exemplo.com')) AS email,
  concat('00', substr(n_cep, 1, 3), '-', substr(n_cep, 4, 3)) AS cep,
  CASE
    WHEN p_municipio < 50 THEN 3550308   -- São Paulo
    WHEN p_municipio < 65 THEN 3518800   -- Guarulhos
    WHEN p_municipio < 80 THEN 3509502   -- Campinas
    WHEN p_municipio < 90 THEN 3516408   -- Franco da Rocha
    ELSE 3550704                         -- São Sebastião
  END AS municipio_id,
  date_add(DATE'1940-01-01', n_nascimento) AS data_nascimento,
  CASE WHEN p_canal < 60 THEN 'app' WHEN p_canal < 90 THEN 'sms' ELSE 'app e sms' END AS canal,
  p_consentimento < 95 AS consentimento,
  CASE WHEN p_consentimento < 95 THEN date_add(DATE'2026-10-01', n_data) END AS data_consentimento,
  'simulado' AS origem
FROM nomes;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Amostra do dado BRUTO (simulado): é isso que NÃO pode chegar à Ouro
SELECT assinante_id, nome, telefone, email, cep, municipio_id, data_nascimento, canal, consentimento
FROM workspace.belka_seguro.prata_assinante
ORDER BY assinante_id
LIMIT 5;

assinante_id,nome,telefone,email,cep,municipio_id,data_nascimento,canal,consentimento
1,Gabriela Dias,(00) 94229-6605,gabriela.dias1@exemplo.com,00928-872,3550308,1957-11-09,app,true
2,Otávio Costa,(00) 93904-0880,otavio.costa2@exemplo.com,00877-710,3550308,1949-05-14,app,true
3,João Moreira,(00) 95077-2681,joao.moreira3@exemplo.com,00426-196,3550308,1964-12-17,app,true
4,Larissa Barbosa,(00) 96242-0322,larissa.barbosa4@exemplo.com,00214-307,3550308,1942-04-13,sms,true
5,Thiago Araújo,(00) 92046-0680,thiago.araujo5@exemplo.com,00201-512,3509502,1940-07-25,sms,false


## 3. O que cada campo vira na Ouro

| Campo bruto | Na Ouro | Técnica | Por quê (LGPD) |
|---|---|---|---|
| telefone | `assinante_hash` + `telefone_mascarado` | hash SHA-256 com salt + máscara `(00) 9****-**12` | pseudonimização: permite contar, deduplicar e cruzar sem ver o número |
| e-mail | `email_mascarado` | máscara `a***@exemplo.com` | basta para conferência visual pelo suporte |
| nome | **não entra** | eliminação | o alerta não precisa do nome (necessidade, art. 6º, III) |
| CEP | **não entra**, fica só o município | generalização | os níveis são por município; o CEP identificaria a rua |
| data de nascimento | `faixa_etaria` e `prioridade_idoso` | generalização | a Belka só precisa saber quem tem 60 anos ou mais (manual de marca: prioridade a idosos) |
| sem consentimento | **não entra** | filtro | sem base legal para usar o dado (art. 7º, I) |

O telefone completo continua só no schema seguro, porque é necessário para **enviar** o SMS. A camada
analítica (Ouro) responde “quantos, onde e quem tem prioridade” sem conter o número.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.belka.ouro_dim_assinante
COMMENT 'Belka · assinantes de alerta (SIMULADOS) protegidos: hash com salt e máscara. Sem nome, CEP, e-mail completo ou nascimento.'
AS
WITH salt AS (
  SELECT salt FROM workspace.belka_seguro.config_hash WHERE versao = 'v1'
),
base AS (
  SELECT *,
         regexp_replace(telefone, '[^0-9]', '') AS tel_digitos,
         floor(months_between(current_date(), data_nascimento) / 12) AS idade
  FROM workspace.belka_seguro.prata_assinante
  WHERE consentimento
)
SELECT
  sha2(concat(s.salt, b.tel_digitos), 256) AS assinante_hash,
  concat('(', substr(b.tel_digitos, 1, 2), ') ', substr(b.tel_digitos, 3, 1), '****-**', right(b.tel_digitos, 2))
                                            AS telefone_mascarado,
  concat(left(b.email, 1), '***@', substring_index(b.email, '@', -1)) AS email_mascarado,
  b.municipio_id,
  CASE WHEN b.idade >= 60 THEN '60 ou mais'
       WHEN b.idade >= 40 THEN '40 a 59'
       ELSE '18 a 39' END                   AS faixa_etaria,
  b.idade >= 60                             AS prioridade_idoso,
  b.canal,
  b.data_consentimento,
  'v1'                                      AS versao_hash
FROM base b
CROSS JOIN salt s;

num_affected_rows,num_inserted_rows


## 4. Antes e depois
A mesma pessoa simulada no cadastro bruto (esquerda) e na Ouro (direita).

In [0]:
%sql
WITH salt AS (SELECT salt FROM workspace.belka_seguro.config_hash WHERE versao = 'v1')
SELECT
  a.nome, a.telefone, a.email, a.cep, a.data_nascimento,
  '→'                                   AS vira,
  concat(left(o.assinante_hash, 16), '…') AS assinante_hash,
  o.telefone_mascarado, o.email_mascarado, o.municipio_id, o.faixa_etaria
FROM workspace.belka_seguro.prata_assinante a
CROSS JOIN salt s
JOIN workspace.belka.ouro_dim_assinante o
  ON o.assinante_hash = sha2(concat(s.salt, regexp_replace(a.telefone, '[^0-9]', '')), 256)
ORDER BY a.assinante_id
LIMIT 5;

nome,telefone,email,cep,data_nascimento,vira,assinante_hash,telefone_mascarado,email_mascarado,municipio_id,faixa_etaria
Gabriela Dias,(00) 94229-6605,gabriela.dias1@exemplo.com,00928-872,1957-11-09,→,487e6a928f98ae40…,(00) 9****-**05,g***@exemplo.com,3550308,60 ou mais
Otávio Costa,(00) 93904-0880,otavio.costa2@exemplo.com,00877-710,1949-05-14,→,d9495b46e645c7ef…,(00) 9****-**80,o***@exemplo.com,3550308,60 ou mais
João Moreira,(00) 95077-2681,joao.moreira3@exemplo.com,00426-196,1964-12-17,→,012604ed32be36f8…,(00) 9****-**81,j***@exemplo.com,3550308,60 ou mais
Larissa Barbosa,(00) 96242-0322,larissa.barbosa4@exemplo.com,00214-307,1942-04-13,→,cd9c9492ad2802bd…,(00) 9****-**22,l***@exemplo.com,3550308,60 ou mais
Natália Rocha,(00) 98273-0306,natalia.rocha6@exemplo.com,00645-170,1955-03-28,→,674c4b574d78e783…,(00) 9****-**06,n***@exemplo.com,3550308,60 ou mais


## 5. Provas da proteção

In [0]:
%sql
-- Contagens: quem entrou na Ouro e quem ficou de fora por falta de consentimento
SELECT
  (SELECT COUNT(*)                    FROM workspace.belka_seguro.prata_assinante)      AS cadastro_bruto,
  (SELECT COUNT_IF(NOT consentimento) FROM workspace.belka_seguro.prata_assinante)      AS sem_consentimento,
  (SELECT COUNT(*)                    FROM workspace.belka.ouro_dim_assinante)          AS na_ouro,
  (SELECT COUNT(DISTINCT assinante_hash) FROM workspace.belka.ouro_dim_assinante)       AS hashes_distintos;

cadastro_bruto,sem_consentimento,na_ouro,hashes_distintos
1000,51,949,949


In [0]:
%sql
-- Colunas da Ouro: não há nome, CEP, e-mail completo, telefone completo nem data de nascimento
DESCRIBE TABLE workspace.belka.ouro_dim_assinante;

col_name,data_type,comment
assinante_hash,string,null
telefone_mascarado,string,null
email_mascarado,string,null
municipio_id,int,null
faixa_etaria,string,null
prioridade_idoso,boolean,null
canal,string,null
data_consentimento,date,null
versao_hash,string,null
# Clustering Information,,


In [0]:
%sql
-- Varredura: nenhuma linha da Ouro tem telefone completo, e-mail completo ou CEP. Tudo deve dar 0.
SELECT
  COUNT_IF(telefone_mascarado RLIKE '[0-9]{4}-[0-9]{4}')  AS telefone_completo,
  COUNT_IF(email_mascarado NOT LIKE '_***@%')             AS email_completo,
  COUNT_IF(concat_ws(' ', telefone_mascarado, email_mascarado) RLIKE '[0-9]{5}-[0-9]{3}') AS cep
FROM workspace.belka.ouro_dim_assinante;

telefone_completo,email_completo,cep
0,0,0


In [0]:
%sql
-- Por que o salt importa: com o salt o hash do cadastro é encontrado na Ouro;
-- sem o salt (como faria quem tentasse força bruta) nenhum hash bate.
WITH salt AS (SELECT salt FROM workspace.belka_seguro.config_hash WHERE versao = 'v1'),
tentativas AS (
  SELECT
    sha2(concat(s.salt, regexp_replace(a.telefone, '[^0-9]', '')), 256) AS hash_com_salt,
    sha2(regexp_replace(a.telefone, '[^0-9]', ''), 256)                 AS hash_sem_salt
  FROM workspace.belka_seguro.prata_assinante a
  CROSS JOIN salt s
  WHERE a.consentimento
)
SELECT
  COUNT_IF(hash_com_salt IN (SELECT assinante_hash FROM workspace.belka.ouro_dim_assinante)) AS encontrados_com_salt,
  COUNT_IF(hash_sem_salt IN (SELECT assinante_hash FROM workspace.belka.ouro_dim_assinante)) AS encontrados_sem_salt
FROM tentativas;

encontrados_com_salt,encontrados_sem_salt
949,0


## 6. Visão agregada para o produto
O painel da Belka e a Defesa Civil não precisam de linha por pessoa: precisam de **quantos assinantes há em cada
município**. A view agrega a dimensão protegida e **suprime grupos com menos de 5 pessoas** (aparecem como nulo),
para que ninguém seja reidentificado por estar sozinho num grupo pequeno.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.belka.ouro_vw_assinantes_municipio
COMMENT 'Belka · assinantes por município (agregado, grupos < 5 suprimidos)'
AS
SELECT
  m.municipio_id,
  m.municipio_nome,
  CASE WHEN COUNT(a.assinante_hash) >= 5 THEN COUNT(a.assinante_hash) END                 AS assinantes,
  CASE WHEN COUNT_IF(a.prioridade_idoso) >= 5 THEN COUNT_IF(a.prioridade_idoso) END         AS assinantes_60_mais,
  CASE WHEN COUNT_IF(a.canal IN ('sms', 'app e sms')) >= 5
       THEN COUNT_IF(a.canal IN ('sms', 'app e sms')) END                                   AS recebem_sms
FROM workspace.belka.ouro_dim_municipio m
LEFT JOIN workspace.belka.ouro_dim_assinante a ON a.municipio_id = m.municipio_id
GROUP BY m.municipio_id, m.municipio_nome;

In [0]:
%sql
SELECT * FROM workspace.belka.ouro_vw_assinantes_municipio ORDER BY assinantes DESC;

municipio_id,municipio_nome,assinantes,assinantes_60_mais,recebem_sms
3550308,São Paulo,474,200,210
3518800,Guarulhos,159,72,67
3509502,Campinas,135,54,50
3550704,São Sebastião,100,31,43
3516408,Franco da Rocha,81,33,30


## 7. Consulta com a Ouro protegida: quantos assinantes cada dia de alerta alcançaria
Cruza a fato de risco com os assinantes sem tocar em nenhum dado pessoal: só hash, município, faixa e canal.
Para cada dia, mostra quantos assinantes estavam em municípios com algum alerta (nível ≥ 1), quantos têm
60 anos ou mais e quantos dependem de SMS (celular simples).

In [0]:
%sql
WITH dia AS (
  SELECT f.municipio_id, t.data, MAX(f.nivel_id) AS nivel_max_dia
  FROM workspace.belka.ouro_fato_risco_hora f
  JOIN workspace.belka.ouro_dim_tempo t ON t.tempo_id = f.tempo_id
  GROUP BY f.municipio_id, t.data
)
SELECT
  date_format(d.data, 'dd/MM/yyyy')                                   AS dia,
  COUNT_IF(d.nivel_max_dia >= 1)                                      AS assinantes_alertados,
  COUNT_IF(d.nivel_max_dia >= 1 AND a.prioridade_idoso)               AS idosos_alertados,
  COUNT_IF(d.nivel_max_dia >= 1 AND a.canal IN ('sms', 'app e sms'))  AS alertas_por_sms
FROM dia d
JOIN workspace.belka.ouro_dim_assinante a ON a.municipio_id = d.municipio_id
GROUP BY d.data
ORDER BY d.data;

dia,assinantes_alertados,idosos_alertados,alertas_por_sms
25/10/2026,949,390,400
26/10/2026,949,390,400
27/10/2026,316,118,123
28/10/2026,135,54,50
29/10/2026,0,0,0
30/10/2026,733,303,320
31/10/2026,0,0,0
01/11/2026,100,31,43
02/11/2026,159,72,67
03/11/2026,100,31,43


## Controle de acesso (em produção)
Na Free Edition há um só usuário, então o controle de acesso fica documentado. Num workspace da equipe seria:

```sql
-- Analistas e o MVP leem só a Ouro protegida
GRANT USE SCHEMA ON SCHEMA workspace.belka TO `analistas`;
GRANT SELECT ON TABLE workspace.belka.ouro_vw_assinantes_municipio TO `analistas`;
-- Ninguém além do serviço de envio de alertas lê o schema seguro
REVOKE ALL PRIVILEGES ON SCHEMA workspace.belka_seguro FROM `account users`;
```